In [1]:
import json
import httpx
import time
import re
import pandas as pd

In [2]:
client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=15.0, write=15.0, pool=15.0),
    limits=httpx.Limits(max_keepalive_connections=0, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers={
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
    },
)

SINA_REFERER_HEADERS = {"Referer": "https://vip.stock.finance.sina.com.cn/mkt/"}


### 新浪行情中心
对应地址 - https://vip.stock.finance.sina.com.cn/mkt/

该行情地址通过`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData?page=1&num=40&sort=symbol&asc=1&node=sh_a&symbol=&_s_r_a=init`（GET）请求行情数据。`page`和`num`分别控制页码和每页数量，`sort`控制排序字段，与`asc`配合控制排序方向（`1`升序、`0`降序）；`node`参数尤其重要，用于筛选不同分类的行情数据。`symbol`通常传空字符串，`_s_r_a`是新浪页面内部参数，有点像request action，比如page代表是翻页触发的请求，init是页面第一次加载等等。

`node`实际的取值范围可参照`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodes`。`hs_a`、`sh_a`、`sz_a`、`hs_bjs`、`cyb`、`kcb`就分别对应沪深 A 股整体、沪 A、深 A、北交所、创业板、科创板，构建 A 股标的池时优先使用这些节点。`hs_s`表示沪深指数集合，`etf_hq_fund` 适合获取场内 ETF 基金行情，是构建 ETF 标的池的首选节点，同时 LOF 基金使用`lof_hq_fund`。

对于接口调用的response数据中，symbol 是带交易所前缀的证券标识（如 bj920000、sh510010），code 是六位证券代码，name 是名称，trade 是最新成交价，pricechange 是相对昨收的涨跌额，changepercent 是涨跌幅（单位为 %），buy 和 sell 分别是当前最优买价和卖价，settlement 是昨收/前一结算价，open、high、low 分别是今开、最高价和最低价，volume 是成交量（股票为股、ETF 为份额），amount 是成交额（元），ticktime 是行情更新时间，per 是市盈率，pb 是市净率；比如安徽凤凰作为股票，这两个字段分别为 18.833 和 1.82，而 180 治理 ETF 返回 0，通常表示不适用或未提供，mktcap 是总市值、nmc 是流通市值，按该接口口径通常以万元计，turnoverratio 是换手率（单位为 %）。

In [3]:
HQ_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData"
HQ_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCount"

In [4]:
HS_A_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'hs_a',
    "symbol": "",
    "_s_r_a": "page",
}

HS_A_COUNT_PARAMS={
    "node": 'hs_a',
}

hs_a_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=HS_A_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())

ha_s_resp = client.request(method="GET", url=HQ_ENDPOINT, params=HS_A_PARAMS, headers=SINA_REFERER_HEADERS)
ha_s_resp_json = ha_s_resp.json()
ha_s_resp_json[0]

{'symbol': 'bj920000',
 'code': '920000',
 'name': '安徽凤凰',
 'trade': '13.490',
 'pricechange': -0.1,
 'changepercent': -0.736,
 'buy': '13.460',
 'sell': '13.490',
 'settlement': '13.590',
 'open': '13.510',
 'high': '13.560',
 'low': '13.310',
 'volume': 177281,
 'amount': 2385976,
 'ticktime': '11:14:36',
 'per': 18.736,
 'pb': 1.811,
 'mktcap': 123676.32,
 'nmc': 77694.204825,
 'turnoverratio': 0.30781}

In [5]:
ETF_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'etf_hq_fund',
    "symbol": "",
    "_s_r_a": "page",
}

ETF_COUNT_PARAMS={
    "node": 'etf_hq_fund',
}

etf_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_counts}")

etf_resp = client.request(method="GET", url=HQ_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_resp_json = etf_resp.json()
etf_resp_json[0]

ETF counts: 1644


{'symbol': 'sh510010',
 'code': '510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': -0.017,
 'changepercent': -0.959,
 'buy': '1.764',
 'sell': '1.774',
 'settlement': '1.773',
 'open': '1.756',
 'high': '1.756',
 'low': '1.756',
 'volume': 600,
 'amount': 1054,
 'ticktime': '11:14:23',
 'per': 0,
 'pb': 0,
 'mktcap': 23271.2779672,
 'nmc': 22920.08464,
 'turnoverratio': 0.00046}

In [6]:
HQ_SIMPLE_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeDataSimple"
HQ_SIMPLE_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCountSimple"

etf_simple_counts = int(client.request(method="GET", url=HQ_SIMPLE_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_simple_counts}")

etf_simple_resp = client.request(method="GET", url=HQ_SIMPLE_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_simple_resp_json = etf_simple_resp.json()
etf_simple_resp_json[0]

ETF counts: 1645


{'symbol': 'sh510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': '-0.017',
 'changepercent': '-0.959',
 'buy': '1.764',
 'sell': '1.774',
 'settlement': '1.773',
 'open': '1.756',
 'high': '1.756',
 'low': '1.756',
 'volume': 600,
 'amount': 1054,
 'code': '510010',
 'ticktime': '11:14:23',
 'state': '00',
 'statetxt': '正常'}

### 单个标的行情快照

In [7]:
SINGLE_QUOTE_ENDPOINT = "http://hq.sinajs.cn/rn={timestamp}&list={symbol}"
SINGLE_QUOTE_HEADER = {"Referer": "http://finance.sina.com.cn/"}

sina_symbol = "sh515080"
sina_quote_resp = client.get(
    SINGLE_QUOTE_ENDPOINT.format(timestamp=int(time.time()), symbol=sina_symbol),
    headers= SINGLE_QUOTE_HEADER
)

# response header 是 Content-Type: application/javascript; charset=GB18030
print(f"{sina_quote_resp.headers}")
# 所以不是取json()，而是取 text
# 'var hq_str_sh513190="港股通金融ETF华夏,1.800,1.800,1.822,1.833,1.799,1.822,1.823,169558700,309415609.000,76200,1.822,40500,1.821,39500,1.820,95600,1.819,9900,1.818,114200,1.823,651900,1.824,37000,1.825,107500,1.826,42000,1.827,2026-08-26,11:30:00,00,";\n'
# .匹配任意符号，*匹配任意数量,?非贪婪匹配，尽可能少匹配
pattern = rf'hq_str_{sina_symbol}="(.*?)";'
sina_quote_match = re.search(pattern, sina_quote_resp.text)
print((f"Raw Text: {sina_quote_resp.text}"))
print(f"Full Match: {sina_quote_match.group(0)}")
print(f"Quote Snapshot: {sina_quote_match.group(1)}")
# name，今开，昨收，最新价，今日最高，今日最低，买一，卖一，成交量，成交额，买一量，买一价，买二量，买二价，买三量，买三价，买四量，买四价，买五量，买五价，卖一量，卖一价，卖二量，卖二价，卖三量，卖三价，卖四量，卖四价，卖五量，卖五价，日期，时间

Headers({'cache-control': 'no-cache', 'content-length': '184', 'connection': 'Keep-Alive', 'content-type': 'application/javascript; charset=GB18030', 'content-encoding': 'gzip'})
Raw Text: var hq_str_sh515080="中证红利ETF招商,1.603,1.606,1.599,1.606,1.591,1.598,1.600,95362499,152314166.000,1960800,1.598,3591700,1.597,5360400,1.596,450300,1.595,1660200,1.594,1602800,1.600,1873300,1.601,2212700,1.602,870100,1.603,591600,1.604,2026-08-27,11:14:41,00,";

Full Match: hq_str_sh515080="中证红利ETF招商,1.603,1.606,1.599,1.606,1.591,1.598,1.600,95362499,152314166.000,1960800,1.598,3591700,1.597,5360400,1.596,450300,1.595,1660200,1.594,1602800,1.600,1873300,1.601,2212700,1.602,870100,1.603,591600,1.604,2026-08-27,11:14:41,00,";
Quote Snapshot: 中证红利ETF招商,1.603,1.606,1.599,1.606,1.591,1.598,1.600,95362499,152314166.000,1960800,1.598,3591700,1.597,5360400,1.596,450300,1.595,1660200,1.594,1602800,1.600,1873300,1.601,2212700,1.602,870100,1.603,591600,1.604,2026-08-27,11:14:41,00,


### K 线数据接口
对应地址 - "https://money.finance.sina.com.cn/quotes_service/api/json_v2.php/CN_MarketData.getKLineData"

symbol 参数控制查询标的的代码，交易所前缀配合6位代码；scale指示查询的 K 线周期，比如，5分钟用5，15分钟用15，30分钟够用30，60分钟用60，日线用240，周线1200，月线7200；ma指定是否返回移动平均线，如果不需要就是no，否则传入需要的周期整数，特殊的如果需要多均线用“5,10,30"的写法。最终 datalen指定返回需要多少根K线，结果从旧到新排列。

In [8]:
KLINE_ENDPOINT = "https://money.finance.sina.com.cn/quotes_service/api/json_v2.php/CN_MarketData.getKLineData"
KLINE_HEADERS = {"Referer": "https://finance.sina.com.cn/"}

kline_symbol = "sh515080"
kline_date = "2026-08-25"
kline_resp = client.get(
    KLINE_ENDPOINT,
    params={
        "symbol": kline_symbol,
        "scale": 240,
        "ma": "no",
        "datalen": 100,
    },
    headers=KLINE_HEADERS,
)

kline_rows = kline_resp.json()
kline_rows


[{'day': '2026-04-02',
  'open': '1.602',
  'high': '1.610',
  'low': '1.597',
  'close': '1.604',
  'volume': '141836376'},
 {'day': '2026-04-03',
  'open': '1.603',
  'high': '1.604',
  'low': '1.574',
  'close': '1.580',
  'volume': '265010326'},
 {'day': '2026-04-07',
  'open': '1.581',
  'high': '1.586',
  'low': '1.572',
  'close': '1.586',
  'volume': '202932465'},
 {'day': '2026-04-08',
  'open': '1.584',
  'high': '1.598',
  'low': '1.576',
  'close': '1.597',
  'volume': '221522199'},
 {'day': '2026-04-09',
  'open': '1.592',
  'high': '1.596',
  'low': '1.583',
  'close': '1.587',
  'volume': '156184000'},
 {'day': '2026-04-10',
  'open': '1.587',
  'high': '1.596',
  'low': '1.584',
  'close': '1.589',
  'volume': '164489400'},
 {'day': '2026-04-13',
  'open': '1.588',
  'high': '1.592',
  'low': '1.580',
  'close': '1.587',
  'volume': '130481200'},
 {'day': '2026-04-14',
  'open': '1.587',
  'high': '1.593',
  'low': '1.580',
  'close': '1.591',
  'volume': '136669800'},


### K 线复权数据
当前 `CN_MarketData.getKLineData` 没有 `adjust`、`qfq`、`hfq` 或 `fqt` 参数，直接请求得到的是不复权行情。新浪把复权信息放在单独的复权因子文件中：`https://finance.sina.com.cn/realstock/company/<symbol>/qfq.js` 返回前复权因子，`https://finance.sina.com.cn/realstock/company/<symbol>/hfq.js` 返回后复权因子。

因此三种口径的获得方式是：不复权直接使用 `getKLineData` 返回的 OHLC；前复权将原始 OHLC 除以 `qfq.js` 的 `f` 因子；后复权将原始 OHLC 乘以 `hfq.js` 的 `f` 因子。复权因子文件是 JavaScript 变量赋值形式，不是纯 JSON，需要截取 `{...}` 后再解析。因子按日期合并到 K 线，成交量不做价格复权。

In [9]:
SINA_QFQ_FACTOR_ENDPOINT = "https://finance.sina.com.cn/realstock/company/{symbol}/qfq.js"
SINA_HFQ_FACTOR_ENDPOINT = "https://finance.sina.com.cn/realstock/company/{symbol}/hfq.js"

sina_adjust_symbol = "sh600519"
sina_adjust_resp = client.get(
    KLINE_ENDPOINT,
    params={
        "symbol": sina_adjust_symbol,
        "scale": 240,
        "ma": "no",
        "datalen": 5000,
    },
    headers=KLINE_HEADERS,
)
sina_adjust_resp.raise_for_status()
sina_raw_df = pd.DataFrame(sina_adjust_resp.json())
sina_raw_df["day"] = pd.to_datetime(sina_raw_df["day"])
sina_price_columns = ["open", "high", "low", "close"]
for column in sina_price_columns:
    sina_raw_df[column] = pd.to_numeric(sina_raw_df[column], errors="coerce")

def sina_factor_df(symbol: str, adjust: str) -> pd.DataFrame:
    endpoint = {"qfq": SINA_QFQ_FACTOR_ENDPOINT, "hfq": SINA_HFQ_FACTOR_ENDPOINT}[adjust]
    response = client.get(endpoint.format(symbol=symbol), headers=KLINE_HEADERS)
    response.raise_for_status()
    text = response.text
    payload = json.loads(text[text.find("{") : text.rfind("}") + 1])
    factor_df = pd.DataFrame(payload["data"]).rename(
        columns={"d": "day", "f": f"{adjust}_factor"}
    )
    factor_df["day"] = pd.to_datetime(factor_df["day"])
    factor_df[f"{adjust}_factor"] = pd.to_numeric(
        factor_df[f"{adjust}_factor"], errors="coerce"
    )
    return factor_df[["day", f"{adjust}_factor"]].sort_values("day")

sina_factor_data = sina_raw_df[["day"]].copy()
sina_factor_data = sina_factor_data.merge(
    sina_factor_df(sina_adjust_symbol, "qfq"), on="day", how="left"
).merge(
    sina_factor_df(sina_adjust_symbol, "hfq"), on="day", how="left"
).sort_values("day")
sina_factor_data[["qfq_factor", "hfq_factor"]] = sina_factor_data[["qfq_factor", "hfq_factor"]].ffill()

sina_kline_no_adjust = sina_raw_df.copy()
sina_kline_qfq = sina_raw_df.merge(sina_factor_data, on="day", how="left")
sina_kline_hfq = sina_kline_qfq.copy()
for column in sina_price_columns:
    sina_kline_qfq[column] = sina_kline_qfq[column] / sina_kline_qfq["qfq_factor"]
    sina_kline_hfq[column] = sina_kline_hfq[column] * sina_kline_hfq["hfq_factor"]

sina_adjusted_close = pd.DataFrame(
    {
        "day": sina_kline_no_adjust["day"],
        "close_no_adjust": sina_kline_no_adjust["close"],
        "close_qfq": sina_kline_qfq["close"],
        "close_hfq": sina_kline_hfq["close"],
    }
).tail()
sina_adjusted_close

,day,close_no_adjust,close_qfq,close_hfq
4995,2026-08-20,1291.50,1291.50,11471.765115
4996,2026-08-21,1272.83,1272.83,11305.928603
4997,2026-08-24,1304.66,1304.66,11588.658981
4998,2026-08-25,1304.00,1304.00,11582.796523
4999,2026-08-26,1302.80,1302.80,11572.137508


In [10]:
sina_adjust_resp.json()

[{'day': '2005-10-21',
  'open': '46.100',
  'high': '46.450',
  'low': '46.000',
  'close': '46.300',
  'volume': '339538'},
 {'day': '2005-10-24',
  'open': '46.200',
  'high': '47.150',
  'low': '46.200',
  'close': '47.050',
  'volume': '444467'},
 {'day': '2005-10-25',
  'open': '47.330',
  'high': '47.800',
  'low': '47.300',
  'close': '47.640',
  'volume': '1159486'},
 {'day': '2005-10-26',
  'open': '47.620',
  'high': '47.740',
  'low': '47.300',
  'close': '47.580',
  'volume': '963579'},
 {'day': '2005-10-27',
  'open': '47.540',
  'high': '47.540',
  'low': '46.550',
  'close': '46.970',
  'volume': '974815'},
 {'day': '2005-10-28',
  'open': '46.980',
  'high': '48.100',
  'low': '46.980',
  'close': '47.560',
  'volume': '698231'},
 {'day': '2005-10-31',
  'open': '47.490',
  'high': '48.800',
  'low': '47.110',
  'close': '48.190',
  'volume': '668572'},
 {'day': '2005-11-01',
  'open': '48.390',
  'high': '48.550',
  'low': '47.820',
  'close': '48.300',
  'volume': '8

### AkShare、easyquotation 与腾讯复权口径核验
AkShare 的 `stock_zh_a_daily` 并不是给 `CN_MarketData.getKLineData` 传入 `adjust` 参数。当前实现使用新浪 `https://finance.sina.com.cn/realstock/company/<symbol>/hisdata_klc2/klc_kl.js` 获取原始日 K，再分别请求 `https://finance.sina.com.cn/realstock/company/<symbol>/qfq.js` 和 `https://finance.sina.com.cn/realstock/company/<symbol>/hfq.js` 获取复权因子；`adjust=""`、`adjust="qfq"`、`adjust="hfq"` 分别对应不复权、前复权和后复权，另外还可以用 `adjust="qfq-factor"` 或 `adjust="hfq-factor"` 只返回因子。前面使用的 `CN_MarketData.getKLineData` 也是不复权数据，和 `hisdata_klc2/klc_kl.js` 的原始 OHLC 可以直接对齐。
当前版本的 `easyquotation` 主要封装腾讯实时快照 `http://qt.gtimg.cn/q=`，没有新浪 A 股日 K 线的前复权/后复权封装；它的 `daykline` 模块是港股 K 线，不应和新浪 A 股复权接口混用。
因此新浪和腾讯的复权 endpoint 不一样。下面用同一个标的 `sh600519` 对比新浪原始 K 线加因子计算结果，与腾讯 `newfqkline/get` 返回的 `day`、`qfqday`、`hfqday`。比较只看收盘价，避免腾讯旧接口成交量单位为手而新浪返回股造成干扰。

In [ ]:
TENCENT_FQ_KLINE_ENDPOINT = "https://proxy.finance.qq.com/ifzqgtimg/appstock/app/newfqkline/get"
sina_tencent_cross_symbol = "sh600519"
sina_tencent_cross_start_date = "2020-01-01"
sina_tencent_cross_end_date = "2022-12-31"
sina_tencent_cross_count = 640

def tencent_close_df(symbol: str, adjust: str) -> pd.DataFrame:
    response = client.get(
        TENCENT_FQ_KLINE_ENDPOINT,
        params={
            "_var": f"kline_day{adjust}",
            "param": (
                f"{symbol},day,{sina_tencent_cross_start_date},"
                f"{sina_tencent_cross_end_date},{sina_tencent_cross_count},{adjust}"
            ),
        },
    )
    response.raise_for_status()
    response_text = response.text
    payload = json.loads(response_text[response_text.find("{") : response_text.rfind("}") + 1])
    if payload.get("code") != 0:
        raise ValueError(payload)
    data_key = "day" if adjust == "" else f"{adjust}day"
    rows = payload["data"][symbol][data_key]
    close_data = pd.DataFrame(
        {"day": [row[0] for row in rows], "close": [row[2] for row in rows]}
    )
    close_data["day"] = pd.to_datetime(close_data["day"])
    close_data["close"] = pd.to_numeric(close_data["close"], errors="coerce")
    return close_data

sina_cross_data = (
    sina_kline_no_adjust[["day", "close"]]
    .rename(columns={"close": "close_sina_no_adjust"})
    .merge(
        sina_kline_qfq[["day", "close"]].rename(columns={"close": "close_sina_qfq"}),
        on="day",
        how="inner",
    )
    .merge(
        sina_kline_hfq[["day", "close"]].rename(columns={"close": "close_sina_hfq"}),
        on="day",
        how="inner",
    )
    .merge(sina_factor_data, on="day", how="left")
)

for adjustment in ["", "qfq", "hfq"]:
    tencent_close = tencent_close_df(sina_tencent_cross_symbol, adjustment).rename(
        columns={"close": f"close_tencent_{adjustment or 'no_adjust'}"}
    )
    sina_cross_data = sina_cross_data.merge(tencent_close, on="day", how="inner")

comparison_rows = []
for adjustment in ["no_adjust", "qfq", "hfq"]:
    sina_column = f"close_sina_{adjustment}"
    tencent_column = f"close_tencent_{adjustment}"
    absolute_difference = (sina_cross_data[sina_column] - sina_cross_data[tencent_column]).abs()
    relative_difference = absolute_difference / sina_cross_data[tencent_column]
    comparison_rows.append(
        {
            "adjust": adjustment,
            "overlap_rows": len(sina_cross_data),
            "max_abs_close_diff": absolute_difference.max(),
            "max_relative_close_diff": relative_difference.max(),
        }
    )

sina_tencent_adjustment_summary = pd.DataFrame(comparison_rows).round(6)
sina_tencent_adjustment_samples = (
    sina_cross_data[
        sina_cross_data["day"].isin(
            pd.to_datetime(["2022-06-13", "2022-06-30", "2022-12-27"])
        )
    ][
        [
            "day",
            "qfq_factor",
            "hfq_factor",
            "close_sina_no_adjust",
            "close_tencent_no_adjust",
            "close_sina_qfq",
            "close_tencent_qfq",
            "close_sina_hfq",
            "close_tencent_hfq",
        ]
    ]
    .round(2)
)
print(sina_tencent_adjustment_summary)
sina_tencent_adjustment_samples

交叉核验的含义是：如果不复权收盘价一致而 qfq/hfq 不一致，说明不是标的代码或原始 K 线选错，而是两个数据源采用了不同的复权因子或复权算法。因而本 notebook 中的 `sina_kline_qfq` 和 `sina_kline_hfq` 应标注为“新浪复权因子法（AkShare 口径）”，不能直接改名成腾讯复权结果。
这次差异并不是简单的四舍五入误差。以 `sh600519` 的 2022-06-13 为例，新浪因子法是 `1856 / 1.1569675988 = 1604.19`；Yahoo 的 adjusted close 约为 `1604.11`，与新浪因子法接近。腾讯 qfq 则为 `1633.06`，而新浪分红记录中该日之后的现金分红合计约为 `223.0142` 元/股，`1856 - 223.0142 = 1632.99`，与腾讯结果只差约 `0.07`，说明腾讯至少在 qfq 上采用了现金分红做差的另一种口径。
所以 5.58% 的 qfq 差异和 31.20% 的 hfq 差异可以由复权定义不同解释，但这不是可以忽略的小误差：如果需要跨数据源拼接，必须固定一个供应商和一种复权口径。复权值还会随着之后新增的分红、送转事件而变化，复现时应记录请求日期。

### 独立复权收盘价对照
Yahoo Finance 的 `adjclose` 只用于验证收盘价，不作为本项目的生产数据源。对同一时间区间进行对照，可以判断新浪的乘除因子法是否与另一套常见的比例复权口径接近。

In [ ]:
YAHOO_CHART_ENDPOINT = "https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"

def yahoo_adjusted_close_summary(symbol: str) -> pd.DataFrame:
    response = client.get(
        YAHOO_CHART_ENDPOINT.format(symbol=symbol),
        params={
            "period1": int(pd.Timestamp("2020-01-01", tz="UTC").timestamp()),
            "period2": int(pd.Timestamp("2023-01-01", tz="UTC").timestamp()),
            "interval": "1d",
            "events": "div,splits",
            "includeAdjustedClose": "true",
        },
    )
    if response.status_code != 200:
        return pd.DataFrame({"message": [f"Yahoo request skipped: HTTP {response.status_code}"]})
    result = response.json()["chart"]["result"][0]
    yahoo_qfq_data = pd.DataFrame(
        {
            "day": pd.to_datetime(result["timestamp"], unit="s").normalize(),
            "close_yahoo_adj": result["indicators"]["adjclose"][0]["adjclose"],
        }
    )
    sina_yahoo_qfq = sina_kline_qfq[["day", "close"]].rename(
        columns={"close": "close_sina_qfq"}
    ).merge(yahoo_qfq_data, on="day", how="inner")
    sina_yahoo_qfq["abs_diff"] = (
        sina_yahoo_qfq["close_sina_qfq"] - sina_yahoo_qfq["close_yahoo_adj"]
    ).abs()
    sina_yahoo_qfq["relative_diff"] = (
        sina_yahoo_qfq["abs_diff"] / sina_yahoo_qfq["close_yahoo_adj"]
    )
    return pd.DataFrame(
        [
            {
                "overlap_rows": len(sina_yahoo_qfq),
                "max_abs_close_diff": sina_yahoo_qfq["abs_diff"].max(),
                "max_relative_close_diff": sina_yahoo_qfq["relative_diff"].max(),
            }
        ]
    ).round(6)

sina_yahoo_qfq_summary = yahoo_adjusted_close_summary("600519.SS")
sina_yahoo_qfq_summary

如果把两个数据源都理解为同一种 qfq/hfq，31.20% 当然不合理；但本次结果是两种复权定义的差异，不是普通精度误差。新浪/AkShare 的比例因子法与独立的 `adjclose` 对照接近，腾讯 qfq 在这个标的上则接近用未来现金分红做差的口径。下面再用东方财富的 `fqt` 结果进行独立核验。

### 东方财富复权收盘价交叉核验
东方财富个股历史 K 线通过 `https://push2his.eastmoney.com/api/qt/stock/kline/get` 获取，`secid` 使用 `市场号.证券代码`，日线使用 `klt=101`，`fqt=0`、`fqt=1`、`fqt=2` 分别表示不复权、前复权、后复权。这里仍然只比较收盘价，并与上一节的 `sina_cross_data` 使用相同的 `sh600519` 和日期交集；若主机偶发断开，代码按网页的分发方式尝试编号主机。

In [ ]:
EASTMONEY_KLINE_ENDPOINTS = [
    "https://push2his.eastmoney.com/api/qt/stock/kline/get",
    "http://51.push2his.eastmoney.com/api/qt/stock/kline/get",
    "http://35.push2his.eastmoney.com/api/qt/stock/kline/get",
    "http://50.push2his.eastmoney.com/api/qt/stock/kline/get",
]

def eastmoney_close_df(symbol: str, adjust: str) -> pd.DataFrame:
    adjust_code = {"no_adjust": "0", "qfq": "1", "hfq": "2"}[adjust]
    market_code = {"sh": "1", "sz": "0", "bj": "0"}[symbol[:2]]
    params = {
        "fields1": "f1,f2,f3,f4,f5,f6",
        "fields2": "f51,f52,f53,f54,f55,f56,f57,f58,f59,f60,f61",
        "ut": "fa5fd1943c7b386f172d6893dbfba10b",
        "klt": 101,
        "fqt": adjust_code,
        "secid": f"{market_code}.{symbol[2:]}",
        "beg": "20200101",
        "end": "20221231",
        "smplmt": 1000000,
        "lmt": 1000000,
    }
    last_error = None
    for endpoint in EASTMONEY_KLINE_ENDPOINTS:
        request_params = params.copy()
        if endpoint.startswith("http://"):
            request_params["cb"] = f"eastmoney_{adjust}_{int(time.time() * 1000)}"
        try:
            response = client.get(
                endpoint,
                params=request_params,
                headers={"Referer": "https://quote.eastmoney.com/"},
            )
            response.raise_for_status()
            response_text = response.text.strip()
            if response_text.startswith("{"):
                payload = json.loads(response_text)
            else:
                payload = json.loads(
                    response_text[response_text.find("(") + 1 : response_text.rfind(")")]
                )
            rows = (payload.get("data") or {}).get("klines") or []
            if rows:
                break
        except (httpx.HTTPError, ValueError, KeyError, IndexError) as exc:
            last_error = exc
    else:
        raise RuntimeError(
            f"东方财富 fqt={adjust_code} 请求失败，请检查接口限流或主机可用性: {last_error}"
        )
    columns = [
        "day", "open", "close", "high", "low",
        "volume", "amount", "amplitude", "pct",
        "change", "turnover",
    ]
    result = pd.DataFrame([row.split(",") for row in rows], columns=columns)
    result["day"] = pd.to_datetime(result["day"])
    result["close"] = pd.to_numeric(result["close"], errors="coerce")
    return result[["day", "close"]].rename(
        columns={"close": f"close_eastmoney_{adjust}"}
    )

eastmoney_cross_data = sina_cross_data.copy()
for adjustment in ["no_adjust", "qfq", "hfq"]:
    eastmoney_cross_data = eastmoney_cross_data.merge(
        eastmoney_close_df(sina_tencent_cross_symbol, adjustment),
        on="day",
        how="inner",
    )

eastmoney_comparison_rows = []
for adjustment in ["no_adjust", "qfq", "hfq"]:
    eastmoney_column = f"close_eastmoney_{adjustment}"
    for source in ["sina", "tencent"]:
        source_column = f"close_{source}_{adjustment}"
        absolute_difference = (
            eastmoney_cross_data[source_column] - eastmoney_cross_data[eastmoney_column]
        ).abs()
        relative_difference = absolute_difference / eastmoney_cross_data[eastmoney_column]
        eastmoney_comparison_rows.append(
            {
                "source": source,
                "adjust": adjustment,
                "overlap_rows": len(eastmoney_cross_data),
                "max_abs_close_diff": absolute_difference.max(),
                "max_relative_close_diff": relative_difference.max(),
            }
        )

eastmoney_adjustment_summary = pd.DataFrame(eastmoney_comparison_rows).round(6)
eastmoney_hfq_ratio = (
    eastmoney_cross_data["close_eastmoney_hfq"]
    / eastmoney_cross_data["close_tencent_hfq"]
)
eastmoney_hfq_ratio_summary = pd.DataFrame(
    [
        {
            "mean_ratio_eastmoney_to_tencent": eastmoney_hfq_ratio.mean(),
            "min_ratio": eastmoney_hfq_ratio.min(),
            "max_ratio": eastmoney_hfq_ratio.max(),
        }
    ]
).round(8)
eastmoney_adjustment_samples = eastmoney_cross_data[
    eastmoney_cross_data["day"].isin(
        pd.to_datetime(["2022-06-13", "2022-06-30", "2022-12-27"])
    )
][
    [
        "day",
        "close_sina_no_adjust",
        "close_eastmoney_no_adjust",
        "close_tencent_no_adjust",
        "close_sina_qfq",
        "close_eastmoney_qfq",
        "close_tencent_qfq",
        "close_sina_hfq",
        "close_eastmoney_hfq",
        "close_tencent_hfq",
    ]
].round(2)
print(eastmoney_adjustment_summary)
print(eastmoney_hfq_ratio_summary)
eastmoney_adjustment_samples

东方财富核验进一步说明：在 `sh600519` 的共同区间 `2020-05-19` 至 `2022-12-30`、共 640 个交易日中，三方不复权收盘价逐日完全一致；东方财富 `fqt=1` 与腾讯 `qfq` 的收盘价逐日完全一致，而新浪因子法在 `2022-06-13` 为 `1604.19`，东方财富和腾讯均为 `1633.06`。因此前复权的 `5.58%` 差异不是腾讯数据解析错误，至少在这个标的和区间内，腾讯与东方财富采用了同一套（或产生完全相同结果的）前复权口径。
后复权在 `2022-06-13` 的结果分别为新浪 `14249.27`、东方财富 `9811.42`、腾讯 `10993.79`。东方财富/腾讯后复权价格比的均值约为 `0.89244`，整个区间约在 `0.892286` 到 `0.892561` 之间，主要表现为不同的基准价格缩放；新浪/东方财富的比值则约在 `1.41999` 到 `1.46996` 之间，不是单纯的固定倍数。
所以目前更准确的结论是：新浪、腾讯、东方财富的原始行情一致，但复权结果不能仅凭 `qfq`/`hfq` 标签互换。腾讯与东方财富的前复权结果在本样本中一致；后复权仍存在供应商基准差异；新浪因子法与另外两者存在实质差异。生产和回测应固定数据源、复权口径及查询日期。